# Synthetic 50 MP camera experiment - PRD v5

Read this first: outputs are hypothetical synthetic captures, never recovered Sony RAW.
The executable MVP is bilinear structure plus analytic inverse ISP, CFA-first electron
noise, 14-bit ADC in uint16 DNG, and independently developed 16-bit TIFF.
HAT/Real-ESRGAN SR accepts verified user checkpoints; no weights are downloaded.
Learned HDR/RAW, extended sensor and optics requests remain unavailable.
Their requests remain visible and never count as executed.
No calibration, recovered spectra or full-scale scientific validation is claimed.
Run only in Colab. Do not execute on the 8 GB development Mac.


In [ ]:
# Colab setup: use a checkout containing this notebook's revision.
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("This notebook is Colab-only; local execution is prohibited")
REPOSITORY = Path("/content/experiment_50MP")
if not (REPOSITORY / "prd_v5/config.py").is_file():
    raise RuntimeError("Place the matching repository checkout at /content/experiment_50MP first")
subprocess.run([sys.executable, "-m", "pip", "install", str(REPOSITORY)], check=True)

## Configuration

All booleans below are requests. Defaults request maximum scale and the full feature
set, while privacy fabrication remains false. Baseline execution requires explicit
opt-in below. It does not fulfill the skipped requests. Do not enable that opt-in
when strict learned execution is required. No checkpoints are downloaded.


In [ ]:
TARGET_CAMERA = "SONY_A1II_LIKE"
GEOMETRY_MODE = "PRESERVE_SOURCE_ASPECT_50MP"

SR_BACKEND = "HAT"
# Supply all SRCheckpoint fields: path, sha256, backend, repository_revision,
# source, licence, licence_accepted. No invented pins or implicit downloads.
SR_CHECKPOINT = None
INVERSE_ISP_MODE = "HYBRID"

ENABLE_HDR = True
# Experimental analytic tone hypothesis requires explicit opt-in; learned HDR stays unavailable.
ACCEPT_ANALYTIC_HDR_EXPERIMENT = False
HDR_MAX_RADIANCE = 16.0
ENABLE_LEARNED_RGB2RAW = True

CFA_PATTERN = "RGGB"
ADC_BITS = 14

ENABLE_SHOT_NOISE = True
ENABLE_READ_NOISE = True

ENABLE_PRNU = True
ENABLE_DSNU = True
ENABLE_ROW_FPN = True
ENABLE_COLUMN_FPN = True
ENABLE_DARK_CURRENT = True
ENABLE_HOT_PIXELS = True
ENABLE_DEAD_PIXELS = True

ENABLE_DISTORTION = True
ENABLE_VIGNETTING = True
ENABLE_CHROMATIC_ABERRATION = True
ENABLE_DIFFRACTION = True

FABRICATE_UNOBSERVABLE_METADATA = False

WRITE_DNG = True
WRITE_TIFF16 = True
WRITE_MANIFEST = True

SEED = 42
NATIVE_FIT_POLICY = "LETTERBOX_SENSOR_MARGIN"
SR_TILE = 512
SR_OVERLAP = 64
RAW_TILE = 1024
RAW_OVERLAP = 128
MODEL_DTYPE = "bf16_if_supported_else_fp16"
CPU_ACCUMULATION = "numpy.memmap"
GLOBAL_GUIDANCE_SCALE = 0.125
TARGET_MEGAPIXELS = 50.0
ALLOW_ANALYTIC_BASELINE = False
ASSUME_UNPROFILED_SRGB = False
SOURCE_2K = "/content/source_2k.png"
SOURCE_4K = "/content/source_4k.png"
OUTPUT = "/content/synthetic_v5"
MAX_DECODE_BYTES = 1073741824
MAX_DEVELOP_BYTES = 8589934592
STRIP_ROWS = 64
# Optional joint heuristic: set all three exposure FIELD_MODE entries to ESTIMATE.
# Example: {"scene_ev100": 4.0, "candidate_count": 8,
#           "electronic_shutter": False, "expanded_iso": False}
# An explicit scene-scale prior, not image metering or calibrated probabilities.
EXPOSURE_PRIOR = None
# Optional shared gray-world proxy: set AsShotNeutral FIELD_MODE to ESTIMATE.
# Rendered-image hypothesis, not recovered capture WB. Null keeps current defaults.
COLOUR_SOURCE_PATH = None
# Optional half-open native TIFF coordinates: (top, left, bottom, right).
# Bounds must fit both renders; no registration or scene correspondence is inferred.
COMPARISON_REGION = None

## Structured field panel

Every PRD sensor, noise, colour, optics and metadata table entry is retained below
with its original prior/constraint text. Edit modes using AUTO, ESTIMATE, FABRICATE
or USER. Null values mean unresolved, not a zero-effect simulation. CFA pattern, ADC depth, black level, full-well capacity, ISO, shutter time,
aperture and random seed resolve into the shared executable camera. Set USER
and supply a value for explicit overrides. AUTO/FABRICATE use the base synthetic
hypothesis; ESTIMATE fails without evidence. Other catalogue edits are rejected.
Read-noise amplitude and ISP hypotheses remain configured separately below. GPS, serials and timestamps remain user-only.


In [ ]:
FIELD_REQUESTS = [
    {
        "name": "Sensor width/height px",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "exact dimensions",
        "prd_contract": [
            "Bayer-active raster",
            "No; configured",
            "target geometry",
            "native profile or aspect-preserving 50 MP",
            "exact dimensions",
        ],
    },
    {
        "name": "Sensor physical size",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "geometry consistency",
        "prd_contract": [
            "silicon imaging area",
            "Not identifiable",
            "camera-profile prior",
            "full-frame profile default: 35.9×24.0 mm for α1 II-like",
            "geometry consistency",
        ],
    },
    {
        "name": "Pixel pitch",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "arithmetic",
        "prd_contract": ["physical sensor pitch", "Derived", "sensor mm ÷ pixels", "profile-derived", "arithmetic"],
    },
    {
        "name": "CFA pattern",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "DNG parse and plane test",
        "prd_contract": [
            "R/G/B photosite arrangement",
            "Not inferable from rendered RGB",
            "profile/user",
            "RGGB default; BGGR/GRBG/GBRG alternatives",
            "DNG parse and plane test",
        ],
    },
    {
        "name": "CFA phase/origin",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "round-trip",
        "prd_contract": ["location of CFA tile origin", "No", "profile", "2×2 phase", "round-trip"],
    },
    {
        "name": "ADC bit depth",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "code range",
        "prd_contract": [
            "converter precision",
            "No",
            "camera profile",
            "12–16 synthetic; 14 default Sony-like",
            "code range",
        ],
    },
    {
        "name": "Black level",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "dark-stat test",
        "prd_contract": [
            "DN corresponding to zero light",
            "No",
            "profile/prior",
            "per-plane sampled constrained positive",
            "dark-stat test",
        ],
    },
    {
        "name": "White level",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "saturation test",
        "prd_contract": ["saturation code", "No", "ADC/profile", "`<2^bits`", "saturation test"],
    },
    {
        "name": "Full-well capacity",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "photon-transfer consistency",
        "prd_contract": [
            "maximum electrons before saturation",
            "No",
            "sensor prior",
            "log-normal camera-family prior",
            "photon-transfer consistency",
        ],
    },
    {
        "name": "Conversion gain",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "PTC consistency",
        "prd_contract": ["electrons per digital code", "No", "derived/prior", "positive", "PTC consistency"],
    },
    {
        "name": "Analogue gain",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "ISO/noise consistency",
        "prd_contract": [
            "pre-ADC signal amplification",
            "Partially tied to ISO, not observable uniquely",
            "exposure hypothesis",
            "piecewise monotonic",
            "ISO/noise consistency",
        ],
    },
    {
        "name": "Digital gain",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "pipeline trace",
        "prd_contract": ["post-ADC scaling", "Not identifiable", "profile", "≈1 by default", "pipeline trace"],
    },
    {
        "name": "Sensor non-linearity",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "flat-field sweep",
        "prd_contract": [
            "pre-ADC deviation from linearity",
            "No",
            "optional LUT",
            "identity by default",
            "flat-field sweep",
        ],
    },
    {
        "name": "Quantisation",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "code histogram",
        "prd_contract": ["ADC mapping", "No", "uniform", "selected bits", "code histogram"],
    },
    {
        "name": "Clipping behaviour",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "saturated pixels",
        "prd_contract": ["saturation handling", "No", "hard clip default", "at white level", "saturated pixels"],
    },
    {
        "name": "Dual conversion gain",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "ISO discontinuity test",
        "prd_contract": [
            "multiple gain regimes",
            "No",
            "optional fabricated model",
            "profile-dependent",
            "ISO discontinuity test",
        ],
    },
    {
        "name": "Rolling/global shutter",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "metadata consistency",
        "prd_contract": [
            "sensor readout style",
            "Weak cues only",
            "prior/user",
            "mechanical/electronic synthetic modes",
            "metadata consistency",
        ],
    },
    {
        "name": "Sensor temperature",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "no claim of truth",
        "prd_contract": [
            "sensor thermal state",
            "No",
            "fabricated only",
            "e.g. −10–60 °C experimental prior",
            "no claim of truth",
        ],
    },
    {
        "name": "Shot noise",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "variance vs mean",
        "prd_contract": [
            "No actual noise can be recovered",
            "Poisson electron draw",
            "physically signal-dependent",
            "variance vs mean",
        ],
    },
    {
        "name": "Read noise",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "dark histogram",
        "prd_contract": [
            "At most weakly inferred",
            "Gaussian / heavy-tail electronic prior",
            "positive e⁻ RMS, profile conditioned",
            "dark histogram",
        ],
    },
    {
        "name": "PRNU",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "flat-field spatial statistics",
        "prd_contract": [
            "No",
            "multiplicative pixel-response map",
            "zero-mean small sigma",
            "flat-field spatial statistics",
        ],
    },
    {
        "name": "DSNU",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "dark-frame emulation",
        "prd_contract": ["No", "additive pixel dark-offset map", "small DN/e⁻ distribution", "dark-frame emulation"],
    },
    {
        "name": "Row FPN",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "row PSD",
        "prd_contract": ["No", "row-correlated offset field", "low-amplitude Gaussian/1/f", "row PSD"],
    },
    {
        "name": "Column FPN",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "column PSD",
        "prd_contract": ["No", "column-correlated offset", "low amplitude", "column PSD"],
    },
    {
        "name": "Dark current",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "long-exposure scaling",
        "prd_contract": [
            "No",
            "Poisson thermal electrons × exposure",
            "exposure/temp conditioned",
            "long-exposure scaling",
        ],
    },
    {
        "name": "Hot pixels",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "defect count",
        "prd_contract": ["No", "sparse defect process", "low-rate Bernoulli", "defect count"],
    },
    {
        "name": "Dead pixels",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "defect count",
        "prd_contract": ["No", "sparse zero-response process", "lower-rate Bernoulli", "defect count"],
    },
    {
        "name": "Blooming",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "saturated-source test",
        "prd_contract": [
            "Usually irrelevant for modern still model",
            "optional saturated-charge spread",
            "disabled default",
            "saturated-source test",
        ],
    },
    {
        "name": "Quantisation noise",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "code histogram",
        "prd_contract": ["Derived", "ADC rounding", "determined by bits/gain", "code histogram"],
    },
    {
        "name": "NoiseProfile",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "compare DNG tags to simulation",
        "prd_contract": [
            "Derived",
            "calculate from actual signal/read model",
            "no independent guessing",
            "compare DNG tags to simulation",
        ],
    },
    {
        "name": "Scene-linear RGB",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "forward-render consistency",
        "prd_contract": [
            "Partly",
            "inverse transfer + inverse tone/HDR reconstruction",
            "non-negative float",
            "forward-render consistency",
        ],
    },
    {
        "name": "Camera-native RGB",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "re-render ΔE / LPIPS",
        "prd_contract": [
            "Ambiguous",
            "analytic inverse colour transform + learned residual",
            "matrix-conditioned",
            "re-render ΔE / LPIPS",
        ],
    },
    {
        "name": "Sensor spectral sensitivity",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "spectral benchmark only",
        "prd_contract": [
            "**No unique inversion**",
            "low-dimensional profile hypothesis only",
            "smooth positive curves if experimental",
            "spectral benchmark only",
        ],
    },
    {
        "name": "ColorMatrix1/2",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "USER: one precomposed 3x3 D65 XYZ-to-camera matrix for both tags; no dual illuminants",
        "validation": "DNG render",
        "prd_contract": [
            "Approximate",
            "constrained camera-profile solve or profile prior",
            "invertible 3×3",
            "DNG render",
        ],
    },
    {
        "name": "ForwardMatrix1/2",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "round trip",
        "prd_contract": ["Derived/profile", "camera→XYZ calibration transform", "DNG-constrained", "round trip"],
    },
    {
        "name": "CameraCalibration1/2",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "profile validity",
        "prd_contract": ["Optional", "identity/default unless calibrated", "close to identity", "profile validity"],
    },
    {
        "name": "CalibrationIlluminant1/2",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "WB interpolation",
        "prd_contract": ["Estimate/prior", "profile illuminants", "typically warm/daylight pair", "WB interpolation"],
    },
    {
        "name": "AnalogBalance",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "DNG parser",
        "prd_contract": ["Not uniquely inferable", "profile prior", "near-unity positive vector", "DNG parser"],
    },
    {
        "name": "AsShotNeutral",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "rendered WB",
        "prd_contract": ["Yes, uncertain", "colour-constancy / WB estimator", "positive, G-normalised", "rendered WB"],
    },
    {
        "name": "Colour temperature",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "held-out EXIF",
        "prd_contract": [
            "Approximate",
            "illuminant estimator",
            "**2500–9900 K** allowed by α1 II official range for Sony-like preset.",
            "held-out EXIF",
        ],
    },
    {
        "name": "Tint",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "neutral-patch test",
        "prd_contract": ["Approximate", "residual chromaticity fit", "bounded", "neutral-patch test"],
    },
    {
        "name": "BaselineExposure",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "developer response",
        "prd_contract": [
            "Derived",
            "chosen RAW headroom/development mapping",
            "plausible EV offset",
            "developer response",
        ],
    },
    {
        "name": "BaselineNoise",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "metadata",
        "prd_contract": ["Derived/profile", "noise-model relative level", "positive", "metadata"],
    },
    {
        "name": "BaselineSharpness",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "metadata",
        "prd_contract": ["Configured", "render profile", "positive", "metadata"],
    },
    {
        "name": "ISO",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Low–medium",
        "prd_contract": ["noise/exposure posterior", "50–102400 synthetic search space", "Low–medium"],
    },
    {
        "name": "Shutter time",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Low",
        "prd_contract": ["motion-blur + exposure posterior", "1/32000 s–30 s", "Low"],
    },
    {
        "name": "Aperture",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Low",
        "prd_contract": ["defocus/DOF + exposure posterior", "f/1–f/22", "Low"],
    },
    {
        "name": "Exposure compensation",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Low",
        "prd_contract": ["rendered brightness vs metering hypothesis", "−5…+5 EV", "Low"],
    },
    {
        "name": "Focal length",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Medium–low",
        "prd_contract": ["image FOV/intrinsics + sensor size", "profile-conditioned", "Medium–low"],
    },
    {
        "name": "35 mm equivalent",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Medium",
        "prd_contract": ["derived", "exact after focal/sensor hypothesis", "Medium"],
    },
    {
        "name": "Focus distance",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Low",
        "prd_contract": ["monocular depth/defocus", "0.2 m–∞", "Low"],
    },
    {
        "name": "Lens model",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Very low",
        "prd_contract": ["candidate or synthetic virtual lens", "blank by default", "Very low"],
    },
    {
        "name": "Lens make",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "None",
        "prd_contract": ["profile/user", "synthetic label", "None"],
    },
    {
        "name": "Radial distortion",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Medium in architectural scenes",
        "prd_contract": ["line geometry fit", "near zero to modest", "Medium in architectural scenes"],
    },
    {
        "name": "Tangential distortion",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Low",
        "prd_contract": ["geometry fit", "near zero", "Low"],
    },
    {
        "name": "Vignetting",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Low",
        "prd_contract": ["radial low-frequency fit after scene-mask", "0–2 EV edge fall-off", "Low"],
    },
    {
        "name": "Lateral CA",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Medium",
        "prd_contract": ["channel displacement around edges", "subpixel–few px at final scale", "Medium"],
    },
    {
        "name": "Longitudinal CA",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Very low",
        "prd_contract": ["depth-dependent coloured blur", "optional fabricated", "Very low"],
    },
    {
        "name": "PSF",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Medium",
        "prd_contract": ["edge spread estimation", "spatially varying optional", "Medium"],
    },
    {
        "name": "Diffraction",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "depends on aperture uncertainty",
        "prd_contract": ["derived from aperture/wavelength", "optical calculation", "depends on aperture uncertainty"],
    },
    {
        "name": "Lens MTF",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "High uncertainty",
        "prd_contract": ["approximate from PSF", "inferred, not manufacturer MTF", "High uncertainty"],
    },
    {
        "name": "Motion blur",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "Medium",
        "prd_contract": ["PSF/kernel estimate", "local/global", "Medium"],
    },
    {
        "name": "Stabilisation",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "None",
        "prd_contract": ["cannot reliably infer", "unspecified/fabricated", "None"],
    },
    {
        "name": "Make",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "`Synthetic`",
        "prd_contract": ["`Synthetic`"],
    },
    {
        "name": "Model",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "selected synthetic profile",
        "prd_contract": ["selected synthetic profile"],
    },
    {
        "name": "UniqueCameraModel",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "explicit synthetic identifier",
        "prd_contract": ["explicit synthetic identifier"],
    },
    {
        "name": "ImageDescription",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "mandatory synthetic-capture disclosure",
        "prd_contract": ["mandatory synthetic-capture disclosure"],
    },
    {
        "name": "Software",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "notebook version + git hash",
        "prd_contract": ["notebook version + git hash"],
    },
    {
        "name": "DNGVersion",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "writer-controlled",
        "prd_contract": ["writer-controlled"],
    },
    {
        "name": "DNGBackwardVersion",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "writer-controlled",
        "prd_contract": ["writer-controlled"],
    },
    {
        "name": "PhotometricInterpretation",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "CFA",
        "prd_contract": ["CFA"],
    },
    {
        "name": "SamplesPerPixel",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "1",
        "prd_contract": ["1"],
    },
    {
        "name": "BitsPerSample",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "sensor container/ADC representation",
        "prd_contract": ["sensor container/ADC representation"],
    },
    {
        "name": "Compression",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "uncompressed or lossless DNG",
        "prd_contract": ["uncompressed or lossless DNG"],
    },
    {
        "name": "CFARepeatPatternDim",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "2×2",
        "prd_contract": ["2×2"],
    },
    {
        "name": "CFAPattern",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "selected Bayer layout",
        "prd_contract": ["selected Bayer layout"],
    },
    {
        "name": "CFAPlaneColor",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "derived",
        "prd_contract": ["derived"],
    },
    {
        "name": "CFALayout",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "derived",
        "prd_contract": ["derived"],
    },
    {
        "name": "ActiveArea",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "generated raster",
        "prd_contract": ["generated raster"],
    },
    {
        "name": "DefaultCropOrigin",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "geometry policy",
        "prd_contract": ["geometry policy"],
    },
    {
        "name": "DefaultCropSize",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "image-supported crop",
        "prd_contract": ["image-supported crop"],
    },
    {
        "name": "BlackLevelRepeatDim",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "CFA-based",
        "prd_contract": ["CFA-based"],
    },
    {
        "name": "BlackLevel",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "simulator values",
        "prd_contract": ["simulator values"],
    },
    {
        "name": "WhiteLevel",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "simulator saturation",
        "prd_contract": ["simulator saturation"],
    },
    {
        "name": "ExposureTime",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "exposure hypothesis",
        "prd_contract": ["exposure hypothesis"],
    },
    {
        "name": "FNumber",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "exposure hypothesis",
        "prd_contract": ["exposure hypothesis"],
    },
    {
        "name": "FocalLength",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "optics hypothesis",
        "prd_contract": ["optics hypothesis"],
    },
    {
        "name": "LensModel",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "blank or synthetic label",
        "prd_contract": ["blank or synthetic label"],
    },
    {
        "name": "MeteringMode",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "unspecified unless fabricated",
        "prd_contract": ["unspecified unless fabricated"],
    },
    {
        "name": "Flash",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "estimate only with strong evidence",
        "prd_contract": ["estimate only with strong evidence"],
    },
    {
        "name": "FocusMode",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "unspecified/fabricated",
        "prd_contract": ["unspecified/fabricated"],
    },
    {
        "name": "DriveMode",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "unspecified/fabricated",
        "prd_contract": ["unspecified/fabricated"],
    },
    {
        "name": "Timestamp",
        "value": None,
        "units": None,
        "mode": "USER",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "blank unless user supplies or fabrication enabled",
        "prd_contract": ["blank unless user supplies or fabrication enabled"],
    },
    {
        "name": "GPS",
        "value": None,
        "units": None,
        "mode": "USER",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "**blank unless user supplies it**",
        "prd_contract": ["**blank unless user supplies it**"],
    },
    {
        "name": "Camera serial",
        "value": None,
        "units": None,
        "mode": "USER",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "blank",
        "prd_contract": ["blank"],
    },
    {
        "name": "Lens serial",
        "value": None,
        "units": None,
        "mode": "USER",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "blank",
        "prd_contract": ["blank"],
    },
    {
        "name": "Battery",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "blank unless experimental fabrication enabled",
        "prd_contract": ["blank unless experimental fabrication enabled"],
    },
    {
        "name": "Shutter count",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "blank",
        "prd_contract": ["blank"],
    },
    {
        "name": "Orientation",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "observed/derived",
        "prd_contract": ["observed/derived"],
    },
    {
        "name": "Source SHA-256",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "measured",
        "prd_contract": ["measured"],
    },
    {
        "name": "Random seed",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "measured/configured",
        "prd_contract": ["measured/configured"],
    },
    {
        "name": "Checkpoint hashes",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "measured",
        "prd_contract": ["measured"],
    },
    {
        "name": "Synthetic-capture flag",
        "value": None,
        "units": None,
        "mode": "AUTO",
        "provenance": "UNSPECIFIED",
        "confidence": None,
        "posterior": None,
        "method": "Not resolved; catalogue only, not an executable override",
        "validation": "always true",
        "prd_contract": ["always true"],
    },
]
FIELD_MODE = {field["name"]: field["mode"] for field in FIELD_REQUESTS}

In [ ]:
# These are the actual executable analytic hypotheses, shared across both runs.
from prd_v5.analytic import AnalyticISP
from prd_v5.config import AnalyticRunConfig
from prd_v5.hypothesis import CameraHypothesis

CAMERA_VALUES = {
    "black_level": 512,
    "full_well_e": 52000.0,
    "read_noise_e": 3.0,
    "iso": 100,
    "shutter_s": 0.008,
    "aperture": 5.6,
}
ISP_VALUES = {
    "camera_to_srgb": [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]],
    "white_balance": [1.0, 1.0, 1.0],
    "exposure_gain": 1.0,
    "inverse_smoothstep": False,
}

## Hardware and request evidence

Resource budgets are admission estimates, not measured peak RSS. Inspect before running.


In [ ]:
import json
import shutil

import psutil

from prd_v5.runtime import runtime_record

output = Path(OUTPUT)
output.mkdir(parents=True, exist_ok=False)
gpu = (
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True
    )
    if shutil.which("nvidia-smi")
    else None
)
runtime = {
    "versions": runtime_record(),
    "ram_available": psutil.virtual_memory().available,
    "disk_free": shutil.disk_usage(output).free,
    "gpu": gpu.stdout if gpu and gpu.returncode == 0 else "unavailable",
}
(output / "runtime.json").write_text(json.dumps(runtime, indent=2))
requests = {
    k: v for k, v in list(globals().items()) if k.isupper() and isinstance(v, (str, int, float, bool, dict, list))
}
(output / "requested.json").write_text(json.dumps(requests, indent=2, allow_nan=False))
print(json.dumps(runtime, indent=2))

In [ ]:
# Save preflight state before any image I/O. Readiness does not mean executed.
from prd_v5.panel import resolve_sr_panel

sr_checkpoint, sr_state = resolve_sr_panel(SR_BACKEND, SR_CHECKPOINT, allow_baseline=ALLOW_ANALYTIC_BASELINE)
active = {"ENABLE_SHOT_NOISE", "ENABLE_READ_NOISE", "WRITE_DNG", "WRITE_TIFF16", "WRITE_MANIFEST"}
states = {}
for name, requested in requests.items():
    if type(requested) is bool:
        states[name] = {
            "requested": requested,
            "effective": requested and name in active,
            "status": "disabled"
            if not requested
            else "enabled"
            if name in active
            else "skipped because integration unavailable",
        }
states["SR_BACKEND"] = sr_state
states["INVERSE_ISP_MODE"] = {
    "requested": INVERSE_ISP_MODE,
    "effective": "ANALYTIC",
    "status": "experimental baseline only; learned branch not executed",
}
(output / "effective.json").write_text(json.dumps(states, indent=2))
print(json.dumps(states, indent=2))

## Sequential paired execution

This runs 50 MP twice in Colab only, with one immutable camera and ISP. It retains
config, hashes, masks, JSON/HTML evidence and independent LibRaw/TIFF checks.
A failure leaves diagnostic evidence, never a pair completion marker.
Scientific metrics, exposure posteriors, source diagnostic plots, EXR, edit sweeps,
learned detail masks and Adobe conformance are pending, not passed.


In [ ]:
from prd_v5.paired import run_analytic_pair
from prd_v5.panel import resolve_analytic_panel_from_source as resolve_analytic_panel

if not ALLOW_ANALYTIC_BASELINE or INVERSE_ISP_MODE == "LEARNED_RGB2RAW":
    raise RuntimeError(
        "Requested full stack unavailable. Explicitly opt into the analytic baseline or wait for adapters."
    )
if TARGET_CAMERA != "SONY_A1II_LIKE" or not all((WRITE_DNG, WRITE_TIFF16, WRITE_MANIFEST)):
    raise ValueError("Baseline currently requires Sony-like camera and all three outputs")
if runtime["ram_available"] < MAX_DEVELOP_BYTES or runtime["disk_free"] < 30 * 1073741824:
    raise RuntimeError("Insufficient available RAM or disk for declared full-scale budgets")
if (SR_TILE, SR_OVERLAP) != (512, 64):
    raise ValueError("Paired SR currently supports only tile=512, overlap=64")
if sr_checkpoint is not None:
    # Colab-only optional inference dependencies. Checkpoint bytes remain user supplied.
    subprocess.run([sys.executable, "-m", "pip", "install", str(REPOSITORY) + "[sr]"], check=True)
camera = CameraHypothesis(
    seed=SEED,
    adc_bits=ADC_BITS,
    cfa_pattern=CFA_PATTERN,
    shot_noise=ENABLE_SHOT_NOISE,
    read_noise=ENABLE_READ_NOISE,
    **CAMERA_VALUES,
)
camera, isp, camera_selections, isp_selections = resolve_analytic_panel(
    camera,
    AnalyticISP(
        **{
            **ISP_VALUES,
            "hdr_max_radiance": (HDR_MAX_RADIANCE if ENABLE_HDR and ACCEPT_ANALYTIC_HDR_EXPERIMENT else 1.0),
        }
    ),
    FIELD_REQUESTS,
    FIELD_MODE,
    exposure_prior=EXPOSURE_PRIOR,
    colour_source_path=COLOUR_SOURCE_PATH,
    assume_unprofiled_srgb=ASSUME_UNPROFILED_SRGB,
    max_decode_bytes=MAX_DECODE_BYTES,
    strip_rows=STRIP_ROWS,
)
config = AnalyticRunConfig(
    camera=camera,
    isp=isp,
    target_megapixels=TARGET_MEGAPIXELS,
    geometry_mode=GEOMETRY_MODE,
    native_fit_policy=NATIVE_FIT_POLICY,
    assume_unprofiled_srgb=ASSUME_UNPROFILED_SRGB,
    strip_rows=STRIP_ROWS,
    max_decode_bytes=MAX_DECODE_BYTES,
    max_develop_bytes=MAX_DEVELOP_BYTES,
)
result = run_analytic_pair(
    SOURCE_2K,
    SOURCE_4K,
    output / "pair",
    config,
    camera_selections=camera_selections,
    isp_selections=isp_selections,
    sr_checkpoint=sr_checkpoint,
    comparison_region=COMPARISON_REGION,
)
# A completed pair is the execution evidence; failed runs retain preflight state.
states["ENABLE_HDR"] = {
    "requested": ENABLE_HDR,
    "effective": isp.hdr_max_radiance > 1,
    "status": "experimental analytic HDR executed" if isp.hdr_max_radiance > 1 else "disabled pending opt-in",
    "learned_hdr": "unavailable; no selected compatible checkpoint",
    "max_radiance": isp.hdr_max_radiance,
}
states["SR_BACKEND"] = {
    **sr_state,
    "effective": sr_state["planned"],
    "status": "executed; scientific fidelity unvalidated",
}
(output / "effective.json").write_text(json.dumps(states, indent=2))
print(json.dumps(result, indent=2))